# GAM so với từng model, và GAM sai ở tầng nào

Hai notebook `02` và `03` chỉ so GAM với **một** đối thủ là Hybrid GBM. Notebook này mở rộng theo
hai chiều:

**Chiều rộng** — so GAM với **mọi model đã train**, cả nhóm hai tầng lẫn nhóm dự đoán trực tiếp.

**Chiều sâu** — tách GAM thành **hai nhánh** (giá cơ bản và hệ số nhân) rồi so từng nhánh, thay vì
chỉ nhìn giá cuối. Đây là chỗ trả lời được câu *vì sao* GAM thắng ở chuyến dài.

## Vì sao phải tách nhánh

Cả GAM lẫn GBM đều dùng chung kiến trúc `giá = giá cơ bản × hệ số nhân`. Nhìn giá cuối chỉ thấy
tổng của hai nguồn sai số. Tách ra mới biết nên ghép ở đâu: nếu lợi thế của GAM chỉ nằm ở một nhánh
thì ghép cả giá cuối là lãng phí — kéo theo cả nhánh yếu vào.

> ⚠️ Nhóm cố định theo giá thật và quãng đường, giống hệt các notebook khác của tuần 5.

In [ ]:
import warnings, time, sys, json
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
%matplotlib inline

BLUE, ORANGE, GREEN, RED, PURPLE, MUT = ("#0072B2", "#E69F00", "#009E73",
                                         "#D55E00", "#CC79A7", "#666666")
INK = "#222222"
plt.rcParams.update({
    "figure.facecolor": "white", "axes.facecolor": "white",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": .25,
    "savefig.dpi": 150, "savefig.bbox": "tight",
    "font.size": 11.5, "axes.titlesize": 12.5, "axes.labelsize": 11.5,
    "xtick.labelsize": 10, "ytick.labelsize": 10, "legend.fontsize": 10.5,
})
EVAL = Path("../model/evaluation")
DATA = Path("../data/hcm_train_ready.parquet")
HINH = Path("../docs/hinh_anh"); HINH.mkdir(parents=True, exist_ok=True)
KQ   = Path("ket_qua"); KQ.mkdir(exist_ok=True)

In [ ]:
# ══ NHÓM CỐ ĐỊNH — quy tắc chốt cho cả tuần 5 ══════════════════════════
# Mentor tuần 4: "giữ nguyên nhóm chuyến giữa các model. Không nên để mỗi model
# tự chia nhóm theo giá mà chính nó dự đoán."
# => nhóm chia theo GIÁ THẬT và QUÃNG ĐƯỜNG, hai thứ không phụ thuộc model nào.
CAT_GIA = [0, 50e3, 100e3, 150e3, 200e3, 300e3, np.inf]
TEN_GIA = ["<50k", "50–100k", "100–150k", "150–200k", "200–300k", ">300k"]
CAT_KM  = [0, 2, 5, 8, 12, 15, np.inf]
TEN_KM  = ["<2", "2–5", "5–8", "8–12", "12–15", ">15"]

def gan_nhom(d, cot_gia="y", cot_km="km"):
    d = d.copy()
    d["band"] = pd.cut(d[cot_gia], CAT_GIA, labels=TEN_GIA)
    d["kmb"]  = pd.cut(d[cot_km],  CAT_KM,  labels=TEN_KM)
    return d

def mape(p, y):
    p, y = np.asarray(p, float), np.asarray(y, float)
    return float(np.mean(np.abs(p - y) / y))

def boot_hieu(p_moc, p_moi, y, B=2000, seed=7):
    """CI 95% cho (sai số mốc − sai số mới). Dương = phương án mới TỐT HƠN."""
    y = np.asarray(y, float)
    h = np.abs(np.asarray(p_moc, float) - y)/y - np.abs(np.asarray(p_moi, float) - y)/y
    if len(h) < 2:
        return float("nan"), float("nan"), float("nan")
    rng = np.random.default_rng(seed)
    mau = rng.integers(0, len(h), size=(B, len(h)))
    pp = h[mau].mean(axis=1)
    return h.mean(), float(np.percentile(pp, 2.5)), float(np.percentile(pp, 97.5))

def bang_theo_nhom(d, cot_moc, cot_moi, ten_moc="mốc", ten_moi="mới"):
    """MAPE hai phương án trên từng nhóm cố định + CI của chênh lệch."""
    hang = []
    for cot_nhom, nhan in [("kmb", "km"), ("band", "giá thật")]:
        for g, s in d.groupby(cot_nhom, observed=True):
            if len(s) < 30:
                continue
            m, lo, hi = boot_hieu(s[cot_moc], s[cot_moi], s.y)
            hang.append({"Chia theo": nhan, "Nhóm": str(g), "n": len(s),
                         ten_moc: mape(s[cot_moc], s.y), ten_moi: mape(s[cot_moi], s.y),
                         "Chênh (điểm)": m*100, "CI thấp": lo*100, "CI cao": hi*100,
                         "Khác 0": "✔" if (lo > 0 or hi < 0) else ""})
    m, lo, hi = boot_hieu(d[cot_moc], d[cot_moi], d.y)
    hang.append({"Chia theo": "—", "Nhóm": "TOÀN TẬP", "n": len(d),
                 ten_moc: mape(d[cot_moc], d.y), ten_moi: mape(d[cot_moi], d.y),
                 "Chênh (điểm)": m*100, "CI thấp": lo*100, "CI cao": hi*100,
                 "Khác 0": "✔" if (lo > 0 or hi < 0) else ""})
    return pd.DataFrame(hang)

def in_bang(df, cot_mape):
    return df.style.format({**{c: "{:.2%}" for c in cot_mape},
                            "Chênh (điểm)": "{:+.2f}", "CI thấp": "{:+.2f}",
                            "CI cao": "{:+.2f}", "n": "{:,}"}).hide(axis="index")

## 1. Nạp toàn bộ dự đoán

Bốn file, mỗi file một nhóm model. Tất cả đều xếp cùng thứ tự hàng — có kiểm tra bằng `gia_that`.

In [ ]:
ut  = pd.read_parquet(EVAL / "uq_pred_test.parquet").reset_index(drop=True)
gam = pd.read_parquet(EVAL / "pred_gam.parquet").reset_index(drop=True)
cb  = pd.read_parquet(EVAL / "pred_gia_co_ban.parquet")
hs  = pd.read_parquet(EVAL / "pred_heso.parquet", columns=["pred", "algo"])
gd  = pd.read_parquet(EVAL / "pred_gia.parquet", columns=["pred", "algo"])

N = len(ut)
assert np.allclose(gam.gia_that.values, ut.gia_that.values), "pred_gam lệch thứ tự hàng"
for a in ["HistGB", "LightGBM", "XGBoost"]:
    s = cb[cb.algo == a]
    assert len(s) == N and np.allclose(s.target_shown_price.values, ut.gia_that.values), \
        f"pred_gia_co_ban[{a}] lệch thứ tự hàng"
print(f"Khớp {N:,} dòng trên cả 4 file")

CAY = ["HistGB", "LightGBM", "XGBoost"]
m5 = (ut.requested_lag_minutes == 5).values

D = pd.DataFrame({
    "km": ut.quote_distance.values[m5],
    "y":  ut.gia_that.values[m5],
    "bt": ut.base_that.values[m5],          # gia co ban THAT
    "ht": ut.heso_that.values[m5],          # he so nhan THAT
})
# nhanh gia co ban va nhanh he so cua tung thuat toan
for a in CAY:
    D[f"b_{a}"] = cb[cb.algo == a].base_pred.values[m5]
    D[f"h_{a}"] = hs[hs.algo == a].pred.values[m5]
    D[f"tt_{a}"] = gd[gd.algo == a].pred.values[m5]        # du doan TRUC TIEP
D["b_GAM"]  = gam.base_pred.values[m5]
D["h_GAM"]  = gam.heso_pred.values[m5]
D["tt_GAM"] = gam.truc_tiep_pred.values[m5]
D["pers"]   = ut.persistence.values[m5]

MODEL = CAY + ["GAM"]
for a in MODEL:
    D[f"p_{a}"] = D[f"b_{a}"] * D[f"h_{a}"]                # hai tang = nhan lai

D = gan_nhom(D)
print(f"Test độ trễ 5 phút: {len(D):,} chuyến · {len(MODEL)} model × 2 kiến trúc + persistence")

## 2. Bảng so sánh đầy đủ — mọi model

Chín cột dự đoán: 4 model × 2 kiến trúc, cộng persistence làm sàn.

In [ ]:
COT = ([(f"{a} · hai tầng", f"p_{a}") for a in MODEL]
       + [(f"{a} · trực tiếp", f"tt_{a}") for a in MODEL]
       + [("Persistence", "pers")])

r = [{"Model": ten, "MAPE": mape(D[c], D.y),
      "MAE": float(np.mean(np.abs(D[c] - D.y))),
      "So với mốc (điểm)": (mape(D[c], D.y) - mape(D.p_HistGB, D.y)) * 100}
     for ten, c in COT]
TQ = pd.DataFrame(r).sort_values("MAPE").reset_index(drop=True)
TQ.style.format({"MAPE": "{:.2%}", "MAE": "{:,.0f}đ",
                 "So với mốc (điểm)": "{:+.2f}"}).hide(axis="index")

Ba điều đọc được ngay:

- **Kiến trúc quan trọng hơn thuật toán.** Chênh lệch giữa hai tầng và trực tiếp lớn hơn chênh lệch
  giữa các thuật toán trong cùng kiến trúc.
- **Ba thuật toán cây gần như trùng nhau** — chọn cái nào cũng vậy.
- **GAM đứng cuối trong nhóm hai tầng** trên toàn tập. Câu hỏi còn lại là nó thắng ở đâu.

In [ ]:
# ═════════ HÌNH GC1 — xep hang model ═════════
fig, ax = plt.subplots(figsize=(9.5, 5.2))
t = TQ[TQ.Model != "Persistence"].sort_values("MAPE", ascending=False).reset_index(drop=True)
mau = [ORANGE if "GAM" in x else (BLUE if "hai tầng" in x else MUT) for x in t.Model]
ax.barh(range(len(t)), t.MAPE*100, color=mau, alpha=.85)
for i, r_ in t.iterrows():
    ax.text(r_.MAPE*100 + .03, i, f"{r_.MAPE:.2%}", va="center", fontsize=10)
ax.set_yticks(range(len(t))); ax.set_yticklabels(t.Model)
ax.axvline(mape(D.p_HistGB, D.y)*100, color=INK, ls="--", lw=1.5,
           label=f"mốc Hybrid GBM {mape(D.p_HistGB, D.y):.2%}")
ax.set_xlim(14.2, max(t.MAPE)*100 + .35)
ax.set_xlabel("MAPE (%) — trục cắt ở 14,2% để thấy chênh lệch nhỏ")
ax.set_title("GC1 — Tám model trên cùng tập test\n"
             "Cam = GAM · xanh = hai tầng · xám = dự đoán trực tiếp",
             fontweight="bold", fontsize=12.5)
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(HINH / "GC1_xep_hang_model.png")
plt.show()

## 3. GAM so với từng model, trên từng nhóm

Bảng trên là con số trung bình. Câu hỏi của tuần 5 nằm ở nhóm chuyến dài và giá cao, nên phải chia
nhóm — và so GAM với **từng** đối thủ chứ không chỉ với mốc.

In [ ]:
NHOM = [("kmb", ">15"), ("kmb", "12–15"), ("kmb", "8–12"),
        ("band", ">300k"), ("band", "200–300k"), (None, "TOÀN TẬP")]

r = []
for cn, g in NHOM:
    s = D if cn is None else D[D[cn].astype(str) == g]
    if len(s) < 30:
        continue
    row = {"Nhóm": g, "n": len(s), "GAM": mape(s.p_GAM, s.y)}
    for a in CAY:
        m, lo, hi = boot_hieu(s[f"p_{a}"], s.p_GAM, s.y)
        row[a] = mape(s[f"p_{a}"], s.y)
        row[f"GAM − {a}"] = m*100
        row[f"rõ_{a}"] = "✔" if (lo > 0 or hi < 0) else ""
    r.append(row)
SS = pd.DataFrame(r)
cot_mape = ["GAM"] + CAY
SS.style.format({**{c: "{:.2%}" for c in cot_mape},
                 **{f"GAM − {a}": "{:+.2f}" for a in CAY}}).hide(axis="index")

Cột `GAM − <thuật toán>` dương nghĩa là GAM tốt hơn thuật toán đó, đơn vị điểm phần trăm. Cột `rõ_`
đánh dấu ✔ khi khoảng tin cậy 95% loại trừ được 0.

Kết luận cần nhìn: GAM thắng **cả ba** thuật toán cây ở nhóm nào, hay chỉ thắng riêng HistGB?

## 4. Phân rã GAM theo tầng

Đây là phần chính. Cả GAM lẫn ba thuật toán cây đều dùng chung kiến trúc hai tầng, nên so được từng
nhánh một:

$$\text{giá} = \underbrace{\text{giá cơ bản}}_{\text{nhánh 1}} \times \underbrace{\text{hệ số nhân}}_{\text{nhánh 2}}$$

In [ ]:
r = []
for a in MODEL:
    r.append({"Model": a,
              "Giá cuối": mape(D[f"p_{a}"], D.y),
              "Nhánh giá cơ bản": mape(D[f"b_{a}"], D.bt),
              "Nhánh hệ số nhân": mape(D[f"h_{a}"], D.ht)})
PR = pd.DataFrame(r)
PR["Cơ bản so GBM (điểm)"] = (PR["Nhánh giá cơ bản"]
                              - PR.loc[PR.Model == "HistGB", "Nhánh giá cơ bản"].iloc[0]) * 100
PR["Hệ số so GBM (điểm)"]  = (PR["Nhánh hệ số nhân"]
                              - PR.loc[PR.Model == "HistGB", "Nhánh hệ số nhân"].iloc[0]) * 100
PR.style.format({"Giá cuối": "{:.2%}", "Nhánh giá cơ bản": "{:.2%}",
                 "Nhánh hệ số nhân": "{:.2%}", "Cơ bản so GBM (điểm)": "{:+.2f}",
                 "Hệ số so GBM (điểm)": "{:+.2f}"}).hide(axis="index")

### Tỷ trọng sai số của GAM nằm ở tầng nào

Lấy log để tách, giống cách làm với Hybrid: $\log(\hat p/p) = \log(\hat b/b) + \log(\hat m/m)$.

In [ ]:
r = []
for a in MODEL:
    lg = np.log(D[f"p_{a}"] / D.y)
    lb = np.log(D[f"b_{a}"] / D.bt)
    lm = np.log(D[f"h_{a}"] / D.ht)
    tot = lg.var()
    r.append({"Model": a,
              "Tầng giá cơ bản": lb.var()/tot,
              "Tầng hệ số nhân": lm.var()/tot,
              "Tương tác": 2*lb.cov(lm)/tot,
              "Sai lệch tách": float(np.abs(lg - lb - lm).max())})
TT = pd.DataFrame(r)
TT.style.format({"Tầng giá cơ bản": "{:.1%}", "Tầng hệ số nhân": "{:.1%}",
                 "Tương tác": "{:+.1%}", "Sai lệch tách": "{:.1e}"}).hide(axis="index")

## 5. Lợi thế của GAM đến từ nhánh nào

Câu quyết định cho việc ghép ở `03`. So từng nhánh theo quãng đường.

In [ ]:
r = []
for g, s in D.groupby("kmb", observed=True):
    r.append({"Nhóm km": str(g), "n": len(s),
              "Cơ bản · GBM": mape(s.b_HistGB, s.bt),
              "Cơ bản · GAM": mape(s.b_GAM, s.bt),
              "Chênh cơ bản": (mape(s.b_HistGB, s.bt) - mape(s.b_GAM, s.bt))*100,
              "Hệ số · GBM": mape(s.h_HistGB, s.ht),
              "Hệ số · GAM": mape(s.h_GAM, s.ht),
              "Chênh hệ số": (mape(s.h_HistGB, s.ht) - mape(s.h_GAM, s.ht))*100})
NH = pd.DataFrame(r)
NH.style.format({c: "{:.2%}" for c in
                 ["Cơ bản · GBM", "Cơ bản · GAM", "Hệ số · GBM", "Hệ số · GAM"]}
                | {"Chênh cơ bản": "{:+.2f}", "Chênh hệ số": "{:+.2f}"}).hide(axis="index")

In [ ]:
# ═════════ HÌNH GC2 — loi the theo nhanh ═════════
fig, ax = plt.subplots(1, 2, figsize=(14.5, 5))
x = np.arange(len(NH))
for a, cot, tieu_de, mau in [(ax[0], "Chênh cơ bản", "Nhánh giá cơ bản", BLUE),
                             (ax[1], "Chênh hệ số", "Nhánh hệ số nhân", ORANGE)]:
    c = [GREEN if v > 0 else RED for v in NH[cot]]
    a.bar(x, NH[cot], color=c, alpha=.85)
    for i, v in enumerate(NH[cot]):
        a.text(i, v, f"{v:+.2f}", ha="center",
               va="bottom" if v > 0 else "top", fontsize=9.5, fontweight="bold")
    a.axhline(0, color=INK, lw=1.3)
    a.set_xticks(x); a.set_xticklabels(NH["Nhóm km"])
    a.set_xlabel("Quãng đường (km)")
    a.set_ylabel("GAM − GBM (điểm) · dương = GAM tốt hơn")
    a.set_title(tieu_de, fontweight="bold")

fig.suptitle("GC2 — Lợi thế của GAM nằm ở NHÁNH GIÁ CƠ BẢN, và chỉ ở chuyến dài\n"
             "Nhánh hệ số nhân: GAM kém đều ở mọi nhóm, không có xu hướng theo quãng đường",
             fontweight="bold", fontsize=13, y=1.03)
fig.tight_layout()
fig.savefig(HINH / "GC2_loi_the_theo_nhanh.png")
plt.show()

## 6. Hệ quả — nên ghép ở tầng nào

Nếu lợi thế chỉ nằm ở nhánh giá cơ bản thì ghép giá cuối là kéo theo cả nhánh hệ số yếu của GAM một
cách vô ích. Thử bốn cách tổ hợp ở nhóm chuyến dài:

In [ ]:
r = []
for g in [">15", "12–15", "8–12"]:
    s = D[D.kmb.astype(str) == g]
    if len(s) < 30:
        continue
    r.append({"Nhóm": g, "n": len(s),
              "GBM cơ bản × GBM hệ số": mape(s.b_HistGB*s.h_HistGB, s.y),
              "GAM cơ bản × GBM hệ số": mape(s.b_GAM*s.h_HistGB, s.y),
              "GBM cơ bản × GAM hệ số": mape(s.b_HistGB*s.h_GAM, s.y),
              "GAM cơ bản × GAM hệ số": mape(s.b_GAM*s.h_GAM, s.y)})
s = D
r.append({"Nhóm": "TOÀN TẬP", "n": len(s),
          "GBM cơ bản × GBM hệ số": mape(s.b_HistGB*s.h_HistGB, s.y),
          "GAM cơ bản × GBM hệ số": mape(s.b_GAM*s.h_HistGB, s.y),
          "GBM cơ bản × GAM hệ số": mape(s.b_HistGB*s.h_GAM, s.y),
          "GAM cơ bản × GAM hệ số": mape(s.b_GAM*s.h_GAM, s.y)})
CH = pd.DataFrame(r)
CH.style.format({c: "{:.2%}" for c in CH.columns if "×" in c}).hide(axis="index")

In [ ]:
# Kiem dinh: ghep nhanh co ban co tot hon ghep gia cuoi khong
s = D[D.kmb.astype(str) == ">15"]
print("Nhóm >15 km — so hai cách dùng GAM:")
for ten, p in [("ghép giá cuối (GAM cả hai nhánh)", s.b_GAM*s.h_GAM),
               ("chỉ lấy nhánh cơ bản của GAM   ", s.b_GAM*s.h_HistGB)]:
    m, lo, hi = boot_hieu(s.b_HistGB*s.h_HistGB, p, s.y)
    print(f"  {ten}: MAPE {mape(p, s.y):.2%} · "
          f"hơn mốc {m*100:+.2f} điểm [{lo*100:+.2f}, {hi*100:+.2f}]")
m, lo, hi = boot_hieu(s.b_GAM*s.h_GAM, s.b_GAM*s.h_HistGB, s.y)
print(f"\n  Chênh giữa hai cách: {m*100:+.2f} điểm [{lo*100:+.2f}, {hi*100:+.2f}]"
      f"  {'→ khác 0 rõ' if (lo>0 or hi<0) else '→ không phân biệt được'}")

## 7. Lưu kết quả

In [ ]:
TQ.to_csv(KQ / "GC_xep_hang_model.csv", index=False)
SS.to_csv(KQ / "GC_gam_vs_tung_model.csv", index=False)
PR.to_csv(KQ / "GC_phan_ra_theo_tang.csv", index=False)
TT.to_csv(KQ / "GC_ty_trong_sai_so.csv", index=False)
NH.to_csv(KQ / "GC_loi_the_theo_nhanh.csv", index=False)
CH.to_csv(KQ / "GC_to_hop_hai_nhanh.csv", index=False)

s = D[D.kmb.astype(str) == ">15"]
json.dump({"loi_the_o_nhanh_co_ban_>15km":
               (mape(s.b_HistGB, s.bt) - mape(s.b_GAM, s.bt))*100,
           "loi_the_o_nhanh_he_so_>15km":
               (mape(s.h_HistGB, s.ht) - mape(s.h_GAM, s.ht))*100,
           "mape_ghep_gia_cuoi": mape(s.b_GAM*s.h_GAM, s.y),
           "mape_ghep_nhanh_co_ban": mape(s.b_GAM*s.h_HistGB, s.y),
           "mape_moc": mape(s.b_HistGB*s.h_HistGB, s.y)},
          open(KQ / "GC_ket_luan.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)
print("Đã lưu vào", KQ.resolve())

## 8. Kết luận cần điền

1. Trên toàn tập, GAM xếp thứ mấy trong 8 model?
2. Ở nhóm `>15 km`, GAM thắng **cả ba** thuật toán cây hay chỉ thắng HistGB?
3. Lợi thế của GAM nằm ở nhánh nào, và nhánh còn lại kém bao nhiêu?
4. Ghép **chỉ nhánh giá cơ bản** có tốt hơn ghép giá cuối không?

> Nếu câu 4 trả lời là có, cần sửa `03_GHEP_GAM_GBM` để trộn ở mức nhánh giá cơ bản thay vì mức giá
> cuối. Việc này không làm kiến trúc phức tạp thêm — vẫn là một hàm trọng số theo quãng đường, chỉ
> khác chỗ áp dụng.